In [15]:
import path from "node:path";
import os from "node:os";
import fs from "node:fs"
import pl from 'npm:nodejs-polars';


interface ZshHistoryRow {
  id?: number;
  timestamp: number;   // unix seconds
  command: string;
}

function extract() {
  const zshHistoryPath = path.join(
    os.homedir(),
    ".zsh_history"
  );
  const historyFile = fs.readFileSync(zshHistoryPath, "utf8");
  const lines = historyFile.split("\n");
  const rows = []
  for (const line of lines) {
    // format: : 1705071243:0;git status
    if (!line.startsWith(": ")) continue;

    const match = line.match(/^: (\d+):\d+;(.*)$/);
    if (!match) continue;

    rows.push({
      timestamp: Number(match[1]),
      command: match[2].split(" ")[0], // why not the full command? i don't want to mess with secrets and env vars
    });

  }
  return rows
}

function displayTable(df: pl.DataFrame, limit:number = 10) {
  console.log(df.shape)
  console.table(df.head(limit).toRecords());
}


const rows = extract()
const df = pl.DataFrame(rows)
displayTable(df)

{ height: 1842, width: 2 }
┌───────┬────────────┬─────────┐
│ (idx) │ timestamp  │ command │
├───────┼────────────┼─────────┤
│     0 │ 1763493968 │ "vite"  │
│     1 │ 1763493968 │ "npm"   │
│     2 │ 1763493968 │ "vite"  │
│     3 │ 1763493968 │ "npx"   │
│     4 │ 1763493968 │ "npm"   │
│     5 │ 1763493968 │ "npx"   │
│     6 │ 1763493968 │ "ls"    │
│     7 │ 1763493968 │ "npx"   │
│     8 │ 1763493968 │ "npm"   │
│     9 │ 1763493968 │ "npm"   │
└───────┴────────────┴─────────┘


In [23]:
displayTable(
  df
  .groupBy("command")
  .agg(pl.col("timestamp").count().alias("count"))
  .sort("count", true)
)

// web dev be like

{ height: 107, width: 2 }
┌───────┬──────────┬───────┐
│ (idx) │ command  │ count │
├───────┼──────────┼───────┤
│     0 │ "npm"    │   462 │
│     1 │ "git"    │   259 │
│     2 │ "ls"     │   213 │
│     3 │ "cd"     │   155 │
│     4 │ "npx"    │    87 │
│     5 │ "pyenv"  │    60 │
│     6 │ "python" │    42 │
│     7 │ "rm"     │    36 │
│     8 │ "pip"    │    34 │
│     9 │ "pytest" │    29 │
└───────┴──────────┴───────┘


In [39]:
const dfWithDateTime = df.withColumn(
  pl
    .col("timestamp")
    .cast(pl.Float64)
    .mul(1000)
    .cast(pl.Datetime("ms"))        // cast to Polars datetime
    .alias("datetime")
).withColumn(
  pl.col("datetime").dt    
    .hour().alias("hour")
)

console.log(dfWithDateTime.columns)
const aggregated = dfWithDateTime.groupBy(["hour"]).agg(pl.len().alias("count"))

displayTable(aggregated)

[ "timestamp", "command", "datetime", "hour" ]
{ height: 14, width: 2 }
┌───────┬──────┬──────┐
│ (idx) │ hour │ len  │
├───────┼──────┼──────┤
│     0 │   15 │   35 │
│     1 │   21 │   79 │
│     2 │   16 │   72 │
│     3 │   17 │   57 │
│     4 │   22 │   19 │
│     5 │   19 │ 1227 │
│     6 │   13 │   54 │
│     7 │    7 │   10 │
│     8 │   20 │  198 │
│     9 │   18 │   23 │
└───────┴──────┴──────┘


In [ ]:
import * as Plot from "npm:@observablehq/plot";
import { document } from "jsr:@manzt/jupyter-helper";

// Convert our DataFrame to Array<Object>
const records = aggregated.toRecords();

Plot.plot({
  width: 900,
  marginLeft: 50,
  color: { legend: true },
  marks: [
    Plot.barY(
      records,
      Plot.groupX({ y: "count" }, {
        x: "type",
        sort: { x: "-y" },
        fill: (d) => d.public ? "Public domain" : "Copyrighted",
      }),
    ),
    
  ],
  // Provide a custom `document`
  document,
});

: 